# Track 2 – Fraud prediction notebook

Loads the trained ensemble (`model.pkl`), applies the **same feature engineering as in training** and writes one
`prediction` in [0, 1] per input row, in input order.

* Input: `DATATHON_INPUT_PATH` (falls back to `test.csv`). No ID column needed; rows may be shuffled.
* Output: `DATATHON_OUTPUT_PATH` (falls back to `predictions.csv`), a single column `prediction`.
* No training, no internet, deterministic: every row is scored independently, so the result does not depend on
  row order or batch composition, and re-running reproduces the file exactly.

`model.pkl` is a self-contained `FraudEnsemble` object. The training-time feature engineering (account-relative
spend spikes, merchant-relative amounts, velocity, maturity × device interactions, out-of-fold target
encodings), the five members (CatBoost ×2, XGBoost, LightGBM, logistic regression – each averaged over 5 seeds),
the rank-style blend and the operating point all live inside the object, so exactly the transformations the
members were trained on are applied here. Libraries used at prediction time: numpy, pandas, lightgbm, catboost
(the XGBoost trees are evaluated in numpy).

In [ ]:
import os, sys, pickle, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

INPUT_PATH  = os.environ.get("DATATHON_INPUT_PATH",  "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")
MODEL_PATH  = "model.pkl"

FEATURE_COLUMNS = [  # the 10 raw feature columns of the Track 2 dataset, in the original CSV order
    "transaction_amount", "transaction_hour", "merchant_category", "country", "transaction_channel",
    "transactions_last_24h", "spend_last_24h", "account_age", "new_device", "transactions_last_1h",
]
print("python", sys.version.split()[0], "| input:", INPUT_PATH, "| output:", OUTPUT_PATH)

In [ ]:
# 1. load the trained model (plain pickle; joblib.load works as well)
try:
    with open(MODEL_PATH, "rb") as f:
        model = pickle.load(f)
except Exception:
    import joblib
    model = joblib.load(MODEL_PATH)
print(type(model).__name__, "| members:", len(model.blobs_) // len(model.seeds), "| seeds per member:", len(model.seeds))

In [ ]:
# 2. read the test features; tolerate a missing/extra id column, shuffled rows, or a header-less file
X = pd.read_csv(INPUT_PATH)
if not set(FEATURE_COLUMNS).issubset(X.columns):
    n_cols = X.shape[1]
    if n_cols in (len(FEATURE_COLUMNS), len(FEATURE_COLUMNS) + 1):      # header-less file: assign names by position
        names = (["id"] if n_cols == len(FEATURE_COLUMNS) + 1 else []) + FEATURE_COLUMNS
        X = pd.read_csv(INPUT_PATH, header=None, names=names)
    else:
        raise ValueError(f"unexpected columns: {list(X.columns)}")
X = X[FEATURE_COLUMNS]            # drop id or any extra column; order columns as in training
n_rows = len(X)
print("rows:", n_rows, "| columns:", list(X.columns))
X.head()

In [ ]:
# 3. predict (feature engineering runs inside the model object, identical to training) and write the output
scores = model.predict_proba(X)[:, 1]                   # ensemble score in [0, 1]; >= 0.5 means flagged as fraud
scores = np.nan_to_num(scores.astype(float), nan=0.0)   # defensive: the output must not contain NaN
scores = np.clip(scores, 0.0, 1.0)
assert len(scores) == n_rows

out = pd.DataFrame({"prediction": scores})
out.to_csv(OUTPUT_PATH, index=False, float_format="%.10f")
print(f"wrote {OUTPUT_PATH}: {len(out)} rows | min={scores.min():.4f} max={scores.max():.4f} | "
      f"flagged (>=0.5): {(scores >= 0.5).sum()} ({(scores >= 0.5).mean()*100:.2f}%)")
out.head()

In [ ]:
# 4. transparency: the engineered inputs the members were trained on (stored inside the model)
feats = sorted({f for (member, seed), cols in model.feature_names_.items() for f in cols})
print(len(feats), "distinct model inputs:")
print(", ".join(feats))